# Matrix Factorisation on Movie Lens 1M dataset
Dataset from: [Movie Lens 1M Dataset](https://grouplens.org/datasets/movielens/1m/)

In [ ]:
# Third-party
import numpy as np
from sklearn.metrics import explained_variance_score, mean_squared_error, mean_absolute_error, r2_score

# Local
import pandas as pd
from Dataset.ml1m_data_loader import load_and_merge_data, ml_test_train_split
from evals.offline_eval_metrics import OfflineModelEvaluator, OfflineSlateEvaluator, coverage
from models.matrix_factorisation.NMF_matrix_factorisation import NMFMatrixFactorisation

### Load dataset

In [2]:
df = load_and_merge_data()
df_train, df_test = ml_test_train_split(df, test_proportion=0.2)

In [3]:
df_train.head()

,user_id,movie_id,rating,timestamp,title,genres
0,392,587,1,2000-12-08 19:43:44,Ghost (1990),Comedy|Romance|Thriller
1,1449,2053,1,2000-11-28 18:49:33,"Honey, I Blew Up the Kid (1992)",Children's|Comedy|Sci-Fi
2,2856,1290,3,2000-10-25 20:22:00,Some Kind of Wonderful (1987),Drama|Romance
3,808,2279,3,2000-11-28 06:39:46,Urban Legend (1998),Horror|Thriller
4,4480,2138,4,2000-07-31 06:32:45,Watership Down (1978),Animation|Children's|Drama|Fantasy


In [4]:
df_test.head()

,index,user_id,movie_id,rating,timestamp,title,genres
0,460249,2840,409,3,2000-10-26 13:53:28,Above the Rim (1994),Drama
1,506871,3118,3701,3,2000-09-19 22:19:26,Alien Nation (1988),Crime|Drama|Sci-Fi
2,967430,5831,1722,3,2000-05-09 19:53:34,Tomorrow Never Dies (1997),Action|Romance|Thriller
3,847450,5091,1956,5,2000-07-02 16:07:07,Ordinary People (1980),Drama
4,805144,4819,1399,2,2000-07-06 22:35:48,Marvin's Room (1996),Drama


### Matrix Factorisation
Matrix factorisation algorithm applied to Top-N and Similarity (by movie) slates.

i.e. answers the questions: "what are the top N movies for a specific user" and "because someone watched a movie, they should watch"


Using:
- [SKLearn NMF (Non-Negative Matrix Factorisation)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)]
- Similarity code from [here](https://github.com/dinesh-git17/movie_recommendation/tree/main)
- Top-N code from [here](https://medium.com/@quindaly/step-by-step-nmf-example-in-python-9974e38dc9f9)

In [5]:
NMF_model = NMFMatrixFactorisation(df_train, min_ratings=0, n_components=50)

### Evaluation metrics
Offline metrics - adapted from [here](https://github.com/aryan-jadon/Evaluation-Metrics-for-Recommendation-Systems/blob/main/recommenders/evaluation/python_evaluation.py)

Table from [here](https://github.com/recommenders-team/recommenders/blob/main/examples/03_evaluate/evaluation.ipynb)
|Metric|Range|Selection criteria|Limitation|Reference|
|------|-------------------------------|---------|----------|---------|
|RMSE|$> 0$|The smaller the better.|May be biased, and less explainable than MAE|[link](https://en.wikipedia.org/wiki/Root-mean-square_deviation)|
|MAE|$\geq 0$|The smaller the better.|Dependent on variable scale.|[link](https://en.wikipedia.org/wiki/Mean_absolute_error)|
|R2|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Coefficient_of_determination)|
|Explained variance|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Explained_variation)|

In [ ]:
model_eval = OfflineModelEvaluator()

In [7]:
# Extract predicted ratings for every movie in df_train
pred_ratings_train = []
for row in df_train.itertuples(index=False):
    pred_ratings_train.append(NMF_model.V.loc[row.user_id][row.title])

In [8]:
# Extract predicted ratings for every movie in df_test
pred_ratings_test = []
for row in df_test.itertuples(index=False):
    pred_ratings_test.append(NMF_model.V.loc[row.user_id][row.title])

In [ ]:
print("Metrics on training data:")
[print(f"{k}: {v:.2f}") for k, v in model_eval.calculate_metrics(list(df_train.rating), pred_ratings_train).items()]
print("\nMetrics on test data:")
[print(f"{k}: {v:.2f}") for k, v in model_eval.calculate_metrics(list(df_test.rating), pred_ratings_test).items()]

Metrics on training data:
RMSE: 2.54
MAE: 2.24
R Squared: -4.18
Explained variance: -0.35

Metrics on test data:
RMSE: 2.76
MAE: 2.48
R Squared: -5.10
Explained variance: -0.26


[None, None, None, None]

### Apply model

In [13]:
recs = NMF_model.movie_similarity("101 Dalmatians (1961)")
recs

Rescuers, The (1977)              0.999389
Aristocats, The (1970)            0.999041
Three Caballeros, The (1945)      0.998768
Bambi (1942)                      0.998292
Fox and the Hound, The (1981)     0.997274
Sword in the Stone, The (1963)    0.997119
Oliver & Company (1988)           0.997047
All Dogs Go to Heaven (1989)      0.996407
Robin Hood (1973)                 0.996228
American Tail, An (1986)          0.996180
Name: 101 Dalmatians (1961), dtype: float64

In [14]:
recs = NMF_model.movie_similarity("101 Dalmatians (1996)")
recs

Homeward Bound II: Lost in San Francisco (1996)    0.990765
George of the Jungle (1997)                        0.986094
Parent Trap, The (1998)                            0.966587
Anastasia (1997)                                   0.955855
Homeward Bound: The Incredible Journey (1993)      0.953707
Madeline (1998)                                    0.948013
American Tail: Fievel Goes West, An (1991)         0.945087
Paulie (1998)                                      0.944268
Casper (1995)                                      0.941548
Tarzan (1999)                                      0.941175
Name: 101 Dalmatians (1996), dtype: float64

In [15]:
recs = NMF_model.movie_similarity("10 Things I Hate About You (1999)")
recs

She's All That (1999)         0.949237
Simply Irresistible (1999)    0.949040
Never Been Kissed (1999)      0.946938
Bachelor, The (1999)          0.934022
Random Hearts (1999)          0.924820
Story of Us, The (1999)       0.909752
Love Letter, The (1999)       0.898975
Mickey Blue Eyes (1999)       0.895375
Drive Me Crazy (1999)         0.893527
Runaway Bride (1999)          0.848550
Name: 10 Things I Hate About You (1999), dtype: float64

In [16]:
recs = NMF_model.movie_similarity("Young Guns (1988)")
recs

Rambo: First Blood Part II (1985)         0.990526
Iron Eagle II (1988)                      0.985025
Iron Eagle (1986)                         0.979768
Braddock: Missing in Action III (1988)    0.969616
Death Wish 4: The Crackdown (1987)        0.961202
Quick and the Dead, The (1995)            0.955094
Last Man Standing (1996)                  0.953669
Rambo III (1988)                          0.951977
Red Dawn (1984)                           0.948652
Aces: Iron Eagle III (1992)               0.945435
Name: Young Guns (1988), dtype: float64

Thoughts:
* Not recommending sequels
* Not using a test-train split -> this algorithm won't work if the requested movie doesn't exist in the pivot table
* Therefore, can't handle new movies or users

## Top-N movies for user

In [17]:
user_id  = 44
#NMF_model.understand_user_profile(user_id)
rec = NMF_model.user_top_N(user_id)

In [18]:
print(f"Recommendations for user {user_id}:")
rec_df = NMF_model.get_recommend_dataframe(rec)
display(rec_df)

Recommendations for user 44:


,movie_id,title,genres,pred_ratings
0,2161,"NeverEnding Story, The (1984)","[Adventure, Children's, Fantasy]",2.905323
1,2797,Big (1988),"[Comedy, Fantasy]",2.858360
2,589,Terminator 2: Judgment Day (1991),"[Action, Sci-Fi, Thriller]",2.480848
3,2005,"Goonies, The (1985)","[Adventure, Children's, Fantasy]",2.357080
4,3114,Toy Story 2 (1999),"[Animation, Children's, Comedy]",2.326917
5,919,"Wizard of Oz, The (1939)","[Adventure, Children's, Drama, Musical]",2.293189
6,720,Wallace & Gromit: The Best of Aardman Animatio...,[Animation],2.292191
7,457,"Fugitive, The (1993)","[Action, Thriller]",2.106497
8,1080,Monty Python's Life of Brian (1979),[Comedy],2.077959
9,2804,"Christmas Story, A (1983)","[Comedy, Drama]",1.940829


In [19]:
rec_ids = rec_df["movie_id"]
eval = OfflineSlateEvaluator(rec_ids, df_test, user_id, 3.5, "evals/movie_embeddings.pkl")

In [20]:
offline_slate_metrics = eval.calculate_metrics(pred_ratings=rec_df["pred_ratings"], k=10)
for k, v in offline_slate_metrics.items():
    print(f"{k}: {v:.2f}")

user_id: 44.00
precision_at_k: 0.30
recall_at_k: 0.12
f1_at_k: 0.17
ndcg_at_k: 0.70
hit_rate_at_k: 1.00
average_precision: 0.56
intra_list_similarity: 0.38
gini_index: 0.07


### Evaluate for all users in test dataset

In [21]:
# Params:
N = 10
K = 10
min_rating_for_relevance = 3.5
users_to_eval = df_test["user_id"][:500]

In [22]:
results = []
all_recommendations = []
print(f"Evaluating recommendations for {len(users_to_eval)} users")
for user_id in users_to_eval:
    # Generate recommendation slate
    rec = NMF_model.user_top_N(user_id, N)
    rec_df = NMF_model.get_recommend_dataframe(rec)
    rec_ids = rec_df["movie_id"]
    all_recommendations.append(rec_ids)

    # Run offline eval for this slate
    eval = OfflineSlateEvaluator(rec_ids, df_test, user_id, min_rating_for_relevance, "evals/movie_embeddings.pkl")
    results.append(eval.calculate_metrics(pred_ratings=rec_df["pred_ratings"], k=K))

# Calculate overall metrics
df_results = pd.DataFrame(results)
overall_metrics = {
    'precision_at_k': df_results['precision_at_k'].mean(),
    'recall_at_k': df_results['recall_at_k'].mean(),
    'f1_at_k': df_results['f1_at_k'].mean(),
    'ndcg_at_k': df_results['ndcg_at_k'].mean(),
    'hit_rate_at_k': df_results['hit_rate_at_k'].mean(),
    'mean_average_precision': df_results['average_precision'].mean(),
    'mean_intra_list_similarity': df_results['intra_list_similarity'].mean(),
    'mean_gini_index': df_results['gini_index'].mean(),
    'catalog_coverage': coverage(all_recommendations, df_train['movie_id'].nunique()),
    'num_users': len(results),
    'k': K
}
for k, v in overall_metrics.items():
    print(f"{k}: {v:.2f}")

Evaluating recommendations for 500 users
precision_at_k: 0.43
recall_at_k: 0.13
f1_at_k: 0.17
ndcg_at_k: 0.74
hit_rate_at_k: 0.94
mean_average_precision: 0.62
mean_intra_list_similarity: 0.40
mean_gini_index: 0.08
catalog_coverage: 0.15
num_users: 500.00
k: 10.00


## User profile evaluation

|User|Total Ratings|Overall Avg Rating|Overall Std Dev|Must include| Should include| Must exclude|
|------|-------------------------------|---------|----------|---------|----|---|
| 6013 | 124 | 4.08 | 1.23 | Comedy, Drama | Musical, Romance | Action |
| 2195 | 258 | 3.41 | 1.41 | Action, Sci-Fi | Drama | Musical |
| 1198 | 102 | 3.66 | 1.51 | Action | Drama, Thriller | Children's |
| 3662 | 88  | 3.03 | 1.64 | Sci-Fi | Horror | Comedy |
| 4713 | 66  | 3.03 | 1.55 | Drama | Romance | Horror |


In [23]:
user_ids = [6013, 2195, 1198, 3662, 4713]

In [24]:
for user_id in user_ids:
    NMF_model.understand_user_profile(user_id, rating_dist=False, wc=False)
    rec = NMF_model.user_top_N(user_id)
    print(f"Recommendations for user {user_id}")
    display(NMF_model.get_recommend_dataframe(rec))

Recommendations for user 6013


,movie_id,title,genres,pred_ratings
0,1947,West Side Story (1961),"[Musical, Romance]",2.672497
1,3751,Chicken Run (2000),"[Animation, Children's, Comedy]",2.514976
2,3114,Toy Story 2 (1999),"[Animation, Children's, Comedy]",2.370526
3,912,Casablanca (1942),"[Drama, Romance, War]",2.174350
4,1223,"Grand Day Out, A (1992)","[Animation, Comedy]",2.171580
5,920,Gone with the Wind (1939),"[Drama, Romance, War]",2.131481
6,1282,Fantasia (1940),"[Animation, Children's, Musical]",1.851042
7,2355,"Bug's Life, A (1998)","[Animation, Children's, Comedy]",1.785885
8,720,Wallace & Gromit: The Best of Aardman Animatio...,[Animation],1.583248
9,17,Sense and Sensibility (1995),"[Drama, Romance]",1.513855


Recommendations for user 2195


,movie_id,title,genres,pred_ratings
0,2628,Star Wars: Episode I - The Phantom Menace (1999),"[Action, Adventure, Fantasy, Sci-Fi]",3.966854
1,1376,Star Trek IV: The Voyage Home (1986),"[Action, Adventure, Sci-Fi]",2.738043
2,2115,Indiana Jones and the Temple of Doom (1984),"[Action, Adventure]",2.442177
3,1676,Starship Troopers (1997),"[Action, Adventure, Sci-Fi, War]",2.414463
4,32,Twelve Monkeys (1995),"[Drama, Sci-Fi]",2.352397
5,1097,E.T. the Extra-Terrestrial (1982),"[Children's, Drama, Fantasy, Sci-Fi]",2.341530
6,1387,Jaws (1975),"[Action, Horror]",2.266564
7,1396,Sneakers (1992),"[Crime, Drama, Sci-Fi]",2.247052
8,2716,Ghostbusters (1984),"[Comedy, Horror]",2.176940
9,1909,"X-Files: Fight the Future, The (1998)","[Mystery, Sci-Fi, Thriller]",2.172078


Recommendations for user 1198


,movie_id,title,genres,pred_ratings
0,3793,X-Men (2000),"[Action, Sci-Fi]",3.814141
1,1240,"Terminator, The (1984)","[Action, Sci-Fi, Thriller]",3.017829
2,1196,Star Wars: Episode V - The Empire Strikes Back...,"[Action, Adventure, Drama, Sci-Fi, War]",2.857638
3,3510,Frequency (2000),"[Drama, Thriller]",1.828512
4,1080,Monty Python's Life of Brian (1979),[Comedy],1.741725
5,3717,Gone in 60 Seconds (2000),"[Action, Crime]",1.476767
6,480,Jurassic Park (1993),"[Action, Adventure, Sci-Fi]",1.425364
7,2791,Airplane! (1980),[Comedy],1.374396
8,318,"Shawshank Redemption, The (1994)",[Drama],1.337598
9,1291,Indiana Jones and the Last Crusade (1989),"[Action, Adventure]",1.294969


Recommendations for user 3662


,movie_id,title,genres,pred_ratings
0,2288,"Thing, The (1982)","[Action, Horror, Sci-Fi, Thriller]",1.558213
1,1129,Escape from New York (1981),"[Action, Adventure, Sci-Fi, Thriller]",1.537623
2,2455,"Fly, The (1986)","[Horror, Sci-Fi]",1.522143
3,2009,Soylent Green (1973),"[Sci-Fi, Thriller]",1.481452
4,2527,Westworld (1973),"[Action, Sci-Fi, Thriller, Western]",1.478032
5,3471,Close Encounters of the Third Kind (1977),"[Drama, Sci-Fi]",1.394905
6,2528,Logan's Run (1976),"[Action, Adventure, Sci-Fi]",1.388003
7,3703,Mad Max 2 (a.k.a. The Road Warrior) (1981),"[Action, Sci-Fi]",1.326092
8,1258,"Shining, The (1980)",[Horror],1.325529
9,1997,"Exorcist, The (1973)",[Horror],1.272806


Recommendations for user 4713


,movie_id,title,genres,pred_ratings
0,858,"Godfather, The (1972)","[Action, Crime, Drama]",2.233836
1,3793,X-Men (2000),"[Action, Sci-Fi]",1.941703
2,539,Sleepless in Seattle (1993),"[Comedy, Romance]",1.484013
3,593,"Silence of the Lambs, The (1991)","[Drama, Thriller]",1.382222
4,110,Braveheart (1995),"[Action, Drama, War]",1.332775
5,597,Pretty Woman (1990),"[Comedy, Romance]",1.320246
6,3755,"Perfect Storm, The (2000)","[Action, Adventure, Thriller]",1.319845
7,1721,Titanic (1997),"[Drama, Romance]",1.288215
8,3555,U-571 (2000),"[Action, Thriller]",1.217648
9,2671,Notting Hill (1999),"[Comedy, Romance]",1.127509


Based on the requirements outlined in the table above - the NMF MF model passes all qualitative criteria